# 📈 Notebook 04 — Regression: Revenue Forecasting
### TeleConnect · 7 Regressors: LR | Ridge | Lasso | ElasticNet | DT | RF | SVR
> **Fully automated** — just run all cells (Runtime → Run all). No manual steps needed.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install',
    'pandas==2.1.4','numpy==1.26.2','scikit-learn==1.3.2',
    'matplotlib==3.8.2','seaborn==0.13.0','-q'],check=True)

import pandas as pd,numpy as np,matplotlib.pyplot as plt,seaborn as sns
import warnings,os,time,pickle
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV,KFold,train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.preprocessing import LabelEncoder,StandardScaler
from scipy import stats
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi']=100
RANDOM_STATE=42; np.random.seed(RANDOM_STATE)
os.makedirs('reports/figures',exist_ok=True); os.makedirs('models',exist_ok=True)
print('✅ Setup complete')

In [ ]:
# ── Auto-load or rebuild regression data ──────────────────────────────────────
def build_reg_data():
    URL='https://raw.githubusercontent.com/dsrscientist/dataset1/master/Telco-Customer-Churn.csv'
    df=pd.read_csv(URL)
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce').fillna(0)
    y_all=df['MonthlyCharges'].copy()
    service_cols=['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                  'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    df['ServiceCount']=df[service_cols].apply(lambda r:sum(v=='Yes' for v in r),axis=1)
    df['AvgMonthlySpend']=np.where(df['tenure']==0,df['MonthlyCharges'],df['TotalCharges']/df['tenure'])
    df['IsLongTermCustomer']=(df['tenure']>24).astype(int)
    le=LabelEncoder()
    for col in ['gender','Partner','Dependents','PhoneService','PaperlessBilling','Churn']:
        df[col]=le.fit_transform(df[col].astype(str))
    ohe=['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
         'DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaymentMethod']
    df=pd.get_dummies(df,columns=ohe,drop_first=True)
    bc=df.select_dtypes(include='bool').columns; df[bc]=df[bc].astype(int)
    df=df.drop(columns=['customerID','MonthlyCharges'],errors='ignore')
    sc=[c for c in ['tenure','TotalCharges','AvgMonthlySpend','ServiceCount'] if c in df.columns]
    ss=StandardScaler(); df[sc]=ss.fit_transform(df[sc])
    X_r=df.drop(columns=['Churn'],errors='ignore')
    X_tr,X_tmp,y_tr,y_tmp=train_test_split(X_r,y_all,test_size=0.30,random_state=RANDOM_STATE)
    X_v,X_te,y_v,y_te=train_test_split(X_tmp,y_tmp,test_size=0.50,random_state=RANDOM_STATE)
    return X_tr,X_v,X_te,y_tr,y_v,y_te

try:
    X_train=pd.read_csv('data/processed/X_reg_train.csv')
    X_val=pd.read_csv('data/processed/X_reg_val.csv')
    X_test=pd.read_csv('data/processed/X_reg_test.csv')
    y_train=pd.read_csv('data/processed/y_reg_train.csv').squeeze()
    y_val=pd.read_csv('data/processed/y_reg_val.csv').squeeze()
    y_test=pd.read_csv('data/processed/y_reg_test.csv').squeeze()
    print('✅ Loaded preprocessed regression data')
except:
    print('Rebuilding regression data...')
    X_train,X_val,X_test,y_train,y_val,y_test=build_reg_data()
    print('✅ Data rebuilt')
print(f'Train:{X_train.shape} Val:{X_val.shape} Test:{X_test.shape}')
print(f'Target range: ${y_test.min():.2f} — ${y_test.max():.2f}')

In [ ]:
# ── Evaluation helper ─────────────────────────────────────────────────────────
reg_results=[]
CV=KFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)

def adj_r2(r2,n,p): return 1-(1-r2)*(n-1)/(n-p-1) if n-p-1>0 else float('nan')

def evaluate_reg(name,model,t):
    yp=model.predict(X_test)
    n,p=X_test.shape
    mae=mean_absolute_error(y_test,yp); mse=mean_squared_error(y_test,yp)
    rmse=np.sqrt(mse); r2=r2_score(y_test,yp); ar2=adj_r2(r2,n,p)
    print(f'\n{"="*55}\n  {name}\n{"="*55}')
    print(f'  MAE:{mae:.4f}  MSE:{mse:.4f}  RMSE:{rmse:.4f}  R²:{r2:.4f}  Adj R²:{ar2:.4f}  Time:{t:.1f}s')
    fig,ax=plt.subplots(figsize=(6,5))
    ax.scatter(y_test,yp,alpha=0.4,color='steelblue',s=15)
    mn=min(float(y_test.min()),yp.min()); mx=max(float(y_test.max()),yp.max())
    ax.plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect fit')
    ax.set_xlabel('Actual ($)'); ax.set_ylabel('Predicted ($)')
    ax.set_title(f'Actual vs Predicted — {name}\n(R²={r2:.4f}, RMSE={rmse:.2f})')
    ax.legend(); plt.tight_layout()
    sn=name.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(f'reports/figures/04_avp_{sn}.png',bbox_inches='tight'); plt.show()
    reg_results.append({'Model':name,'MAE':round(mae,4),'MSE':round(mse,4),'RMSE':round(rmse,4),
        'R²':round(r2,4),'Adj R²':round(ar2,4),'Train Time(s)':round(t,2),'y_pred':yp})
    return model
print('✅ Helper ready')

In [ ]:
# ── Model 1: Linear Regression ───────────────────────────────────────────────
t0=time.time(); lr=LinearRegression(); lr.fit(X_train,y_train)
evaluate_reg('Linear Regression',lr,time.time()-t0)

In [ ]:
# ── Model 2: Ridge ────────────────────────────────────────────────────────────
t0=time.time()
ridge=RandomizedSearchCV(Ridge(random_state=RANDOM_STATE),{'alpha':[0.001,0.01,0.1,1,10,100,1000]},
    n_iter=7,cv=CV,scoring='neg_root_mean_squared_error',random_state=RANDOM_STATE,n_jobs=-1)
ridge.fit(X_train,y_train); print(f'Best:{ridge.best_params_}')
ridge_best=ridge.best_estimator_
evaluate_reg('Ridge (L2)',ridge_best,time.time()-t0)

In [ ]:
# ── Model 3: Lasso ────────────────────────────────────────────────────────────
t0=time.time()
lasso=RandomizedSearchCV(Lasso(random_state=RANDOM_STATE,max_iter=5000),{'alpha':[0.001,0.01,0.1,1,10,100]},
    n_iter=6,cv=CV,scoring='neg_root_mean_squared_error',random_state=RANDOM_STATE,n_jobs=-1)
lasso.fit(X_train,y_train); print(f'Best:{lasso.best_params_}')
lasso_best=lasso.best_estimator_
print(f'Features zeroed: {np.sum(lasso_best.coef_==0)}/{len(lasso_best.coef_)}')
evaluate_reg('Lasso (L1)',lasso_best,time.time()-t0)

In [ ]:
# ── Model 4: ElasticNet ───────────────────────────────────────────────────────
t0=time.time()
en=RandomizedSearchCV(ElasticNet(random_state=RANDOM_STATE,max_iter=5000),
    {'alpha':[0.01,0.1,1,10],'l1_ratio':[0.1,0.3,0.5,0.7,0.9]},
    n_iter=12,cv=CV,scoring='neg_root_mean_squared_error',random_state=RANDOM_STATE,n_jobs=-1)
en.fit(X_train,y_train); print(f'Best:{en.best_params_}')
en_best=en.best_estimator_
evaluate_reg('ElasticNet',en_best,time.time()-t0)

In [ ]:
# ── Model 5: Decision Tree Regressor ─────────────────────────────────────────
t0=time.time()
dtr=RandomizedSearchCV(DecisionTreeRegressor(random_state=RANDOM_STATE),
    {'max_depth':[3,5,7,10,None],'min_samples_split':[2,5,10,20],'min_samples_leaf':[1,2,5],'criterion':['squared_error','friedman_mse']},
    n_iter=20,cv=CV,scoring='neg_root_mean_squared_error',random_state=RANDOM_STATE,n_jobs=-1)
dtr.fit(X_train,y_train); print(f'Best:{dtr.best_params_}')
dtr_best=dtr.best_estimator_
evaluate_reg('Decision Tree Regressor',dtr_best,time.time()-t0)

In [ ]:
# ── Model 6: Random Forest Regressor ─────────────────────────────────────────
t0=time.time()
rfr=RandomizedSearchCV(RandomForestRegressor(random_state=RANDOM_STATE,n_jobs=-1),
    {'n_estimators':[100,200],'max_depth':[5,10,None],'min_samples_split':[2,5],'max_features':['sqrt','log2']},
    n_iter=12,cv=CV,scoring='neg_root_mean_squared_error',random_state=RANDOM_STATE,n_jobs=-1)
rfr.fit(X_train,y_train); print(f'Best:{rfr.best_params_}')
rfr_best=rfr.best_estimator_
evaluate_reg('Random Forest Regressor',rfr_best,time.time()-t0)

In [ ]:
# ── Model 7: SVR ──────────────────────────────────────────────────────────────
print('Training SVR...')
t0=time.time()
svr=RandomizedSearchCV(SVR(),
    {'C':[0.1,1,10],'epsilon':[0.1,0.5,1.0],'kernel':['rbf','linear'],'gamma':['scale','auto']},
    n_iter=8,cv=KFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE),
    scoring='neg_root_mean_squared_error',random_state=RANDOM_STATE,n_jobs=-1)
svr.fit(X_train,y_train); print(f'Best:{svr.best_params_}')
svr_best=svr.best_estimator_
evaluate_reg('SVR',svr_best,time.time()-t0)

In [ ]:
# ── Final Comparison Table ────────────────────────────────────────────────────
comp=pd.DataFrame([{k:v for k,v in r.items() if k!='y_pred'} for r in reg_results]).set_index('Model')
print('\n'+'='*75+'\nREGRESSION FINAL COMPARISON TABLE\n'+'='*75)
print(comp.to_string())

fig,axes=plt.subplots(1,3,figsize=(18,5))
for ax,m,asc in zip(axes,['RMSE','R²','MAE'],[True,False,True]):
    vals=comp[m].sort_values(ascending=asc)
    best_v=vals.iloc[0]
    colors=['seagreen' if v==best_v else 'steelblue' for v in vals]
    vals.plot(kind='barh',ax=ax,color=colors,edgecolor='black')
    ax.set_title(f'{m} ({"lower" if asc else "higher"} = better)')
plt.suptitle('Regression Models — Performance Comparison',fontsize=13,y=1.01)
plt.tight_layout()
plt.savefig('reports/figures/04_regression_comparison.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Best Model + Residual Analysis ───────────────────────────────────────────
best_name=comp['R²'].idxmax()
print(f'\n🏆 BEST REGRESSION MODEL: {best_name}')
print(comp.loc[best_name])

best_preds=next(r['y_pred'] for r in reg_results if r['Model']==best_name)
residuals=np.array(y_test)-best_preds

fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].hist(residuals,bins=40,color='steelblue',edgecolor='white',alpha=0.8)
axes[0].axvline(0,color='red',linestyle='--',lw=2)
axes[0].set_title(f'Residual Distribution — {best_name}'); axes[0].set_xlabel('Residual ($)')
stats.probplot(residuals,dist='norm',plot=axes[1]); axes[1].set_title('Q-Q Plot')
axes[2].scatter(best_preds,residuals,alpha=0.4,color='darkorange',s=15)
axes[2].axhline(0,color='red',linestyle='--',lw=2)
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Residuals'); axes[2].set_title('Residuals vs Predicted')
plt.suptitle(f'Residual Analysis — {best_name}',fontsize=13)
plt.tight_layout()
plt.savefig('reports/figures/04_residual_analysis.png',bbox_inches='tight'); plt.show()

stat,p=stats.shapiro(residuals[:500])
print(f'Shapiro-Wilk: stat={stat:.4f}, p={p:.6f}')
print('Residuals approximately normal.' if p>0.05 else 'Slight non-normality — acceptable for tree models.')

# Coefficient analysis for Ridge
coef=pd.Series(ridge_best.coef_,index=X_train.columns).sort_values(key=abs,ascending=False)
fig,ax=plt.subplots(figsize=(10,6))
top15=coef.head(15); colors=['seagreen' if v>0 else 'salmon' for v in top15.sort_values()]
top15.sort_values().plot(kind='barh',ax=ax,color=colors,edgecolor='black')
ax.axvline(0,color='black',lw=0.8); ax.set_title('Ridge Regression — Top 15 Coefficients')
plt.tight_layout(); plt.savefig('reports/figures/04_ridge_coefficients.png',bbox_inches='tight'); plt.show()

with open('models/best_regressor.pkl','wb') as f: pickle.dump(rfr_best,f)
comp.to_csv('reports/regression_comparison.csv')
print('✅ Best regressor saved!')